# DeepGenome Ranking Figures

This notebook renders Fig. 2d-f and Supplementary Fig. 10-13 from the
privacy-safe schema-2 aggregate snapshot. It reports bootstrap uncertainty,
inter-rater agreement, expert-panel composition, and assignment balance
without loading private expert identifiers or row-level rankings.

## Reproducibility Contract

`PhytoBench-Gene-for_plot/frozen/` is the plotting source of truth. Its
`provenance.json` records the 10,000-replicate bootstrap configuration, input
lineage, output checksums, and the locked reporting matrix. The plotting path
does not recompute rankings or load private evaluation inputs.

Fig. 2e and Fig. 2f use the primary crossed expert-by-gene bootstrap interval.
The same uncertainty display is retained in Supplementary Fig. 12 and 13.
Rank-position intervals remain available in source data but are not overlaid
on the compositional stacked bars in Fig. 2d or Supplementary Fig. 10.

In [ ]:
import hashlib
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import plotly.offline as offline
from plotly.subplots import make_subplots


SAVE_FIGS = os.getenv("PHYTOMNI_SAVE") == "1"
DATA_DIR = Path("PhytoBench-Gene-for_plot/frozen")
OUTPUT_DIR = Path("output")
if SAVE_FIGS:
    OUTPUT_DIR.mkdir(exist_ok=True)

offline.init_notebook_mode(connected=True)
pio.kaleido.scope.default_format = "pdf"

In [ ]:
MODEL_ORDER = ["Phytomni", "Gemini", "Claude", "OpenAI", "Grok"]
MODEL_LABELS = {
    "Phytomni": "Phytomni",
    "Gemini": "Gemini Deep Research",
    "OpenAI": "ChatGPT Agent mode",
    "Grok": "Grok DeepSearch",
    "Claude": "Claude deep research",
}
RANK_ORDER = ["R1", "R2", "R3", "R4", "R5"]
RANK_COLORS = [
    "rgb(0,112,160)",
    "rgb(63,145,190)",
    "rgb(126,180,214)",
    "rgb(183,214,232)",
    "rgb(225,239,247)",
]
FOCAL_COLOR = "rgb(23,107,135)"
CONTEXT_COLOR = "rgb(184,188,191)"
ELO_FOCAL_COLOR = "rgb(31,113,179)"
ELO_CONTEXT_COLOR = "rgb(208,210,211)"
MODEL_NEUTRAL_COLOR = "rgb(112,118,122)"
ACCENT_COLOR = "rgb(197,151,55)"
DARK_TEXT = "rgb(35,43,47)"
GRID_COLOR = "rgb(222,225,227)"
LINE_WIDTH = 2
PRIMARY_INTERVAL = "crossed_expert_gene"

STRATIFIED_SCOPES = [
    "well_studied",
    "well_studied.rice",
    "well_studied.maize",
    "well_studied.wheat",
    "well_studied.soybean",
    "well_studied.arabidopsis",
    "uncharacterized",
    "uncharacterized.rice",
    "uncharacterized.maize",
    "uncharacterized.wheat",
    "uncharacterized.soybean",
    "uncharacterized.arabidopsis",
    "rice",
    "maize",
    "wheat",
    "soybean",
    "arabidopsis",
]

TABLE_FILENAMES = {
    "rank_distribution": "rank_distribution.tsv",
    "pl_scores": "pl_scores.tsv",
    "pl_pairwise": "pl_pairwise.tsv",
    "pl_scores_ci": "pl_scores_ci.tsv",
    "rank_distribution_ci": "rank_distribution_ci.tsv",
    "pl_pairwise_ci": "pl_pairwise_ci.tsv",
    "fleiss_kappa": "fleiss_kappa.tsv",
    "kendall_by_gene": "kendall_by_gene.tsv",
    "ordinal_agreement_summary": "ordinal_agreement_summary.tsv",
    "top1_consensus": "top1_consensus.tsv",
    "expert_panel_summary": "expert_panel_summary.tsv",
    "assignment_summary": "assignment_summary.tsv",
}
EXPECTED_ROWS = {
    "rank_distribution": 450,
    "pl_scores": 90,
    "pl_pairwise": 450,
    "pl_scores_ci": 270,
    "rank_distribution_ci": 450,
    "pl_pairwise_ci": 360,
    "fleiss_kappa": 58,
    "kendall_by_gene": 200,
    "ordinal_agreement_summary": 18,
    "top1_consensus": 54,
    "expert_panel_summary": 44,
    "assignment_summary": 18,
}

## Load and Validate Frozen Results

The validation cell locks schema version, table coverage, output checksums,
model coverage, scope coverage, bootstrap size, and privacy-safe columns before
any figure is drawn.

In [ ]:
tables = {
    name: pd.read_csv(DATA_DIR / filename, sep="	")
    for name, filename in TABLE_FILENAMES.items()
}
provenance = json.loads((DATA_DIR / "provenance.json").read_text())

if provenance.get("schema_version") != 2:
    raise ValueError("The figure notebook requires frozen schema version 2.")
if provenance["bootstrap"]["successful_replicates"] != 10_000:
    raise ValueError("The frozen analysis must contain 10,000 bootstrap replicates.")
if set(provenance["model_columns"]) != set(MODEL_ORDER):
    raise ValueError("Frozen model columns do not match the figure configuration.")
if set(provenance["outputs"]) != set(TABLE_FILENAMES.values()):
    raise ValueError("Frozen provenance does not enumerate all aggregate tables.")

for table_name, frame in tables.items():
    if len(frame) != EXPECTED_ROWS[table_name]:
        raise ValueError(
            f"{table_name} has {len(frame)} rows; expected {EXPECTED_ROWS[table_name]}."
        )
    private_columns = {
        "Expert",
        "Expert_ID",
        "AnonymousExpertID",
    } & set(frame.columns)
    if private_columns:
        raise ValueError(
            f"{table_name} exposes private identifier columns: {sorted(private_columns)}"
        )

for filename, expected_sha256 in provenance["outputs"].items():
    digest = hashlib.sha256((DATA_DIR / filename).read_bytes()).hexdigest()
    if digest != expected_sha256:
        raise ValueError(f"Checksum mismatch for {filename}.")

rank_data = tables["rank_distribution"]
score_data = tables["pl_scores"]
pairwise_data = tables["pl_pairwise"]
score_ci_data = tables["pl_scores_ci"]
rank_ci_data = tables["rank_distribution_ci"]
pairwise_ci_data = tables["pl_pairwise_ci"]
fleiss_data = tables["fleiss_kappa"]
kendall_data = tables["kendall_by_gene"]
ordinal_data = tables["ordinal_agreement_summary"]
top1_data = tables["top1_consensus"]
panel_data = tables["expert_panel_summary"]
assignment_data = tables["assignment_summary"]

required_scopes = {"overall", *STRATIFIED_SCOPES}
for table_name, frame in {
    "rank_distribution": rank_data,
    "pl_scores": score_data,
    "pl_pairwise": pairwise_data,
    "pl_scores_ci": score_ci_data,
    "rank_distribution_ci": rank_ci_data,
    "pl_pairwise_ci": pairwise_ci_data,
}.items():
    missing_scopes = required_scopes - set(frame["Scope"])
    if missing_scopes:
        raise ValueError(f"{table_name} is missing scopes: {sorted(missing_scopes)}")

rank_sums = rank_data.groupby(["Scope", "Model"])["Fraction"].sum()
if not (rank_sums.sub(1.0).abs() < 1e-12).all():
    raise ValueError("Rank fractions must sum to one for every scope and model.")

primary_scores = score_ci_data[
    score_ci_data["IntervalAnalysis"] == PRIMARY_INTERVAL
]
if len(primary_scores) != len(required_scopes) * len(MODEL_ORDER):
    raise ValueError("The primary score interval matrix is incomplete.")
if len(pairwise_ci_data) != len(required_scopes) * len(MODEL_ORDER) * 4:
    raise ValueError("The off-diagonal pairwise interval matrix is incomplete.")
if set(panel_data["Dimension"]).difference(
    {
        "Species",
        "Country/Region",
        "Institution_type",
        "Current_position",
        "Years_experience",
        "Gender",
        "Research_domains",
        "Study_species",
        "Annotation_experience",
        "Ai_experience",
        "Conflict_interest",
    }
):
    raise ValueError("The expert-panel table contains an unexpected dimension.")

overall_assignment = assignment_data.set_index("Scope").loc["overall"]
assignment_contract = {
    "NExperts": 120,
    "NGenes": 200,
    "NJudgments": 600,
    "MinGenesPerExpert": 5,
    "MaxGenesPerExpert": 5,
    "MinExpertsPerGene": 3,
    "MaxExpertsPerGene": 3,
}
for column, expected in assignment_contract.items():
    if int(overall_assignment[column]) != expected:
        raise ValueError(f"Unexpected overall assignment value for {column}.")

print("Frozen schema: 2")
print("Bootstrap replicates: 10,000")
print(f"Models: {', '.join(MODEL_ORDER)}")
print(f"Reporting scopes: {len(required_scopes)}")
print("Privacy-safe aggregate tables: 12")

## Fig. 2 and Stratified Plotting Functions

The rank composition is unchanged. Pairwise cells show point estimates and
crossed 95% confidence intervals off the diagonal. Score panels show the
primary crossed interval as error bars on the original Elo-style bars.

In [ ]:
def display_labels(models: list[str] | None = None) -> list[str]:
    selected = MODEL_ORDER if models is None else models
    return [MODEL_LABELS[model] for model in selected]


def scope_label(scope: str) -> str:
    return scope.replace(".", " · ").replace("_", " ").title()


def scope_assignment(scope: str) -> pd.Series:
    rows = assignment_data[assignment_data["Scope"] == scope]
    if len(rows) != 1:
        raise ValueError(f"Assignment summary is incomplete for scope {scope!r}.")
    return rows.iloc[0]


def render_figure(figure: go.Figure, file_prefix: str) -> None:
    if SAVE_FIGS:
        figure.write_image(OUTPUT_DIR / f"{file_prefix}.pdf")
        figure.write_image(OUTPUT_DIR / f"{file_prefix}.png")
    else:
        figure.show()


def rank_distribution_figure(scope: str) -> go.Figure:
    scope_data = rank_data[rank_data["Scope"] == scope]
    rank_matrix = (
        scope_data.pivot(index="Model", columns="Rank", values="Fraction")
        .reindex(index=MODEL_ORDER, columns=RANK_ORDER)
    )
    if rank_matrix.isna().any().any():
        raise ValueError(f"Incomplete rank distribution for scope {scope!r}.")

    figure = go.Figure()
    for rank_label, color in zip(RANK_ORDER, RANK_COLORS, strict=True):
        figure.add_trace(
            go.Bar(
                x=display_labels(),
                y=100.0 * rank_matrix[rank_label],
                name=rank_label.replace("R", "Rank "),
                marker_color=color,
                marker_line_color="white",
                marker_line_width=0.5,
                hovertemplate=(
                    "%{x}<br>" + rank_label.replace("R", "Rank ")
                    + ": %{y:.1f}%<extra></extra>"
                ),
            )
        )
    assignment = scope_assignment(scope)
    figure.update_layout(
        title={
            "text": (
                f"Rank-position distribution — {scope_label(scope)}"
                f"<br><sup>n={int(assignment['NJudgments'])} expert–gene rankings; "
                f"{int(assignment['NGenes'])} genes</sup>"
            ),
            "x": 0.02,
        },
        barmode="stack",
        showlegend=True,
        legend={"orientation": "h", "y": 1.02, "x": 0.98, "xanchor": "right"},
        xaxis={
            "showline": True,
            "linewidth": LINE_WIDTH,
            "linecolor": DARK_TEXT,
            "mirror": True,
            "ticks": "outside",
            "tickwidth": LINE_WIDTH,
            "tickangle": -20,
            "automargin": True,
        },
        yaxis={
            "showline": True,
            "linewidth": LINE_WIDTH,
            "linecolor": DARK_TEXT,
            "mirror": True,
            "ticks": "outside",
            "tickwidth": LINE_WIDTH,
            "title_text": "Rankings (%)",
            "range": [0, 100],
            "gridcolor": GRID_COLOR,
        },
        plot_bgcolor="white",
        paper_bgcolor="white",
        font_family="Arial",
        font_color=DARK_TEXT,
        font_size=18,
        width=1200,
        height=1000,
        margin={"l": 100, "r": 50, "t": 150, "b": 180},
    )
    return figure


def pairwise_probability_figure(scope: str) -> go.Figure:
    point_rows = pairwise_data[pairwise_data["Scope"] == scope]
    interval_rows = pairwise_ci_data[pairwise_ci_data["Scope"] == scope]
    probability_matrix = (
        point_rows.pivot(
            index="RowModel",
            columns="ColumnModel",
            values="Probability",
        )
        .reindex(index=MODEL_ORDER, columns=MODEL_ORDER)
    )
    if probability_matrix.shape != (len(MODEL_ORDER), len(MODEL_ORDER)):
        raise ValueError(f"Incomplete pairwise matrix for scope {scope!r}.")
    probability_values = probability_matrix.to_numpy()
    off_diagonal = ~np.eye(len(MODEL_ORDER), dtype=bool)
    if not np.isfinite(probability_values[off_diagonal]).all():
        raise ValueError(f"Nonfinite off-diagonal probability for scope {scope!r}.")
    if not np.isnan(np.diag(probability_values)).all():
        raise ValueError("Pairwise diagonal must be undefined in frozen source data.")
    interval_index = interval_rows.set_index(["RowModel", "ColumnModel"])
    cell_text: list[list[str]] = []
    for row_model in MODEL_ORDER:
        row_text: list[str] = []
        for column_model in MODEL_ORDER:
            if row_model == column_model:
                row_text.append("")
                continue
            probability = float(probability_matrix.loc[row_model, column_model])
            interval = interval_index.loc[(row_model, column_model)]
            if not np.isclose(probability, interval["Probability"], atol=1e-12):
                raise ValueError("Pairwise point estimate and interval table disagree.")
            row_text.append(
                f"{probability:.2f}<br>"
                f"[{interval['CI95Lower']:.2f}, {interval['CI95Upper']:.2f}]"
            )
        cell_text.append(row_text)

    figure = go.Figure(
        go.Heatmap(
            x=display_labels(),
            y=display_labels(),
            z=probability_matrix.to_numpy(),
            text=cell_text,
            texttemplate="%{text}",
            colorscale="TroPic",
            zmin=0,
            zmax=1,
            zmid=0.5,
        )
    )
    figure.update_layout(
        showlegend=False,
        yaxis_scaleanchor="x",
        xaxis={"showline": False, "ticks": "outside"},
        yaxis={"showline": False, "ticks": "outside"},
        plot_bgcolor="white",
        font_family="Arial",
        font_color="rgb(0,0,0)",
        font_size=20,
        width=1080,
        height=1080,
    )
    return figure


def elo_score_figure(
    scope: str,
    y_range: tuple[int, int] = (1000, 2000),
) -> go.Figure:
    scope_scores = (
        score_data[score_data["Scope"] == scope]
        .set_index("Model")
        .reindex(MODEL_ORDER)
    )
    scope_intervals = (
        score_ci_data[
            (score_ci_data["Scope"] == scope)
            & (score_ci_data["IntervalAnalysis"] == PRIMARY_INTERVAL)
        ]
        .set_index("Model")
        .reindex(MODEL_ORDER)
    )
    if scope_scores["Elo"].isna().any():
        raise ValueError(f"Incomplete Elo scores for scope {scope!r}.")
    if scope_intervals[["Estimate", "CI95Lower", "CI95Upper"]].isna().any().any():
        raise ValueError(f"Incomplete score intervals for scope {scope!r}.")
    values = scope_scores["Elo"].to_numpy(dtype=float)
    estimates = scope_intervals["Estimate"].to_numpy(dtype=float)
    if not np.allclose(values, estimates, atol=1e-10):
        raise ValueError("Score point estimates and interval table disagree.")
    lower = scope_intervals["CI95Lower"].to_numpy(dtype=float)
    upper = scope_intervals["CI95Upper"].to_numpy(dtype=float)
    colors = [
        ELO_FOCAL_COLOR,
        *([ELO_CONTEXT_COLOR] * (len(MODEL_ORDER) - 1)),
    ]

    figure = go.Figure(
        go.Bar(
            x=display_labels(),
            y=values,
            marker_color=colors,
            marker_line_color="rgb(0,0,0)",
            marker_line_width=LINE_WIDTH,
            text=[f"{value:.0f}" for value in values],
            textposition="outside",
            error_y={
                "type": "data",
                "symmetric": False,
                "array": upper - values,
                "arrayminus": values - lower,
                "color": "rgb(0,0,0)",
                "thickness": 2,
                "width": 8,
            },
        )
    )
    figure.update_layout(
        showlegend=False,
        xaxis={
            "showline": True,
            "linewidth": LINE_WIDTH,
            "linecolor": "rgb(0,0,0)",
            "mirror": True,
            "ticks": "outside",
            "tickwidth": LINE_WIDTH,
        },
        yaxis={
            "showline": True,
            "linewidth": LINE_WIDTH,
            "linecolor": "rgb(0,0,0)",
            "mirror": True,
            "ticks": "outside",
            "tickwidth": LINE_WIDTH,
            "title_text": "Score",
            "range": list(y_range),
        },
        plot_bgcolor="white",
        font_family="Arial",
        font_color="rgb(0,0,0)",
        font_size=20,
        width=1080,
        height=1080,
    )
    return figure

## Supplementary Fig. 11: Expert Panel and Agreement

Panel a reports all 11 privacy-safe panel dimensions. Research domains and
study species are multi-select and therefore are shown as independent
percentages rather than parts of a 100% total. Panel b displays the 13 locked
primary/secondary Fleiss-κ scopes; the complete 58-scope table remains in the
source-data snapshot. Panel c shows gene-level Kendall's W distributions, and
panel d reports Top-1 consensus composition.

In [ ]:
DIMENSION_LABELS = {
    "Species": "Species",
    "Country/Region": "Country/region",
    "Institution_type": "Institution type",
    "Current_position": "Current position",
    "Years_experience": "Years of experience",
    "Gender": "Gender",
    "Research_domains": "Research domains†",
    "Study_species": "Study species†",
    "Annotation_experience": "Annotation experience",
    "Ai_experience": "AI experience",
    "Conflict_interest": "Conflict of interest",
}
MULTISELECT_DIMENSIONS = {"Research_domains", "Study_species"}
SPECIES_DISPLAY_ORDER = ["Rice", "Wheat", "Maize", "Soybean", "Arabidopsis"]
KAPPA_SCOPE_ORDER = [
    ("overall", "Overall"),
    ("species.rice", "Rice"),
    ("species.wheat", "Wheat"),
    ("species.maize", "Maize"),
    ("species.soybean", "Soybean"),
    ("species.arabidopsis", "Arabidopsis"),
    ("study_status.well_studied", "Well-studied"),
    ("study_status.uncharacterized", "Uncharacterized"),
    ("model.phytomni", MODEL_LABELS["Phytomni"]),
    ("model.gemini", MODEL_LABELS["Gemini"]),
    ("model.claude", MODEL_LABELS["Claude"]),
    ("model.openai", MODEL_LABELS["OpenAI"]),
    ("model.grok", MODEL_LABELS["Grok"]),
]
TOP1_SCOPE_ORDER = [
    ("overall", "Overall", 200),
    ("species.rice", "Rice", 40),
    ("species.wheat", "Wheat", 40),
    ("species.maize", "Maize", 40),
    ("species.soybean", "Soybean", 40),
    ("species.arabidopsis", "Arabidopsis", 40),
    ("study_status.well_studied", "Well-studied", 100),
    ("study_status.uncharacterized", "Uncharacterized", 100),
]
TOP1_PATTERNS = [
    ("unanimous", "Unanimous", "rgb(23,107,135)"),
    ("majority_2_of_3", "2-of-3 majority", "rgb(99,158,184)"),
    ("all_different", "All different", "rgb(210,215,218)"),
]


def expert_agreement_figure() -> go.Figure:
    figure = make_subplots(
        rows=3,
        cols=2,
        specs=[
            [{"rowspan": 3}, {}],
            [None, {}],
            [None, {}],
        ],
        column_widths=[0.56, 0.44],
        row_heights=[0.34, 0.30, 0.36],
        horizontal_spacing=0.16,
        vertical_spacing=0.08,
        subplot_titles=(
            "Expert panel composition",
            "Fleiss-κ across prespecified scopes",
            "Kendall's W across genes",
            "Top-1 consensus across genes",
        ),
    )

    panel_rows = panel_data.copy()
    panel_rows["FigureOrder"] = panel_rows["DisplayOrder"]
    species_dimensions = panel_rows["Dimension"].isin(["Species", "Study_species"])
    species_order = {label: index for index, label in enumerate(SPECIES_DISPLAY_ORDER, 1)}
    canonical_species = species_dimensions & panel_rows["PublicCategory"].isin(species_order)
    panel_rows.loc[canonical_species, "FigureOrder"] = panel_rows.loc[
        canonical_species, "PublicCategory"
    ].map(species_order)
    panel_rows.loc[species_dimensions & ~canonical_species, "FigureOrder"] = (
        len(SPECIES_DISPLAY_ORDER)
        + panel_rows.loc[species_dimensions & ~canonical_species, "DisplayOrder"]
    )
    panel_rows = panel_rows.sort_values(["Dimension", "FigureOrder"], kind="stable")
    dimension_order = list(dict.fromkeys(panel_data["Dimension"]))
    panel_rows = pd.concat(
        [panel_rows[panel_rows["Dimension"] == dimension] for dimension in dimension_order],
        ignore_index=True,
    )
    panel_colors = [
        ACCENT_COLOR
        if dimension in MULTISELECT_DIMENSIONS
        else DARK_TEXT
        if dimension == "Conflict_interest"
        else FOCAL_COLOR
        for dimension in panel_rows["Dimension"]
    ]
    panel_text = [
        f"{percent:.1f}% ({int(count)}/{int(denominator)})"
        for percent, count, denominator in zip(
            panel_rows["Percent"],
            panel_rows["N"],
            panel_rows["DenominatorN"],
            strict=True,
        )
    ]
    figure.add_trace(
        go.Bar(
            x=panel_rows["Percent"],
            y=[
                [DIMENSION_LABELS[value] for value in panel_rows["Dimension"]],
                panel_rows["PublicCategory"].tolist(),
            ],
            orientation="h",
            marker={
                "color": panel_colors,
                "line": {"color": DARK_TEXT, "width": 0.8},
            },
            text=panel_text,
            textposition="outside",
            cliponaxis=False,
            showlegend=False,
            hovertemplate=(
                "%{y}<br>%{x:.1f}% of denominator"
                "<br>Count: %{text}<extra></extra>"
            ),
        ),
        row=1,
        col=1,
    )

    selected_kappa = fleiss_data[
        fleiss_data["AnalysisTier"].isin(["primary", "locked_secondary"])
    ].set_index("ScopeID")
    if len(selected_kappa) != 13:
        raise ValueError("The prespecified Fleiss-κ display must contain 13 scopes.")
    kappa_labels = [label for _, label in KAPPA_SCOPE_ORDER]
    family_colors = {
        "overall": DARK_TEXT,
        "species": FOCAL_COLOR,
        "study_status": ACCENT_COLOR,
        "model": MODEL_NEUTRAL_COLOR,
    }
    family_symbols = {
        "overall": "diamond",
        "species": "circle",
        "study_status": "square",
        "model": "circle",
    }
    for scope_id, label in KAPPA_SCOPE_ORDER:
        row_data = selected_kappa.loc[scope_id]
        estimate = float(row_data["FleissKappa"])
        lower = float(row_data["CILower"])
        upper = float(row_data["CIUpper"])
        family = str(row_data["ScopeFamily"])
        figure.add_trace(
            go.Scatter(
                x=[estimate],
                y=[label],
                mode="markers",
                marker={
                    "size": 12,
                    "color": family_colors[family],
                    "symbol": family_symbols[family],
                    "line": {"color": DARK_TEXT, "width": 1},
                },
                error_x={
                    "type": "data",
                    "symmetric": False,
                    "array": [upper - estimate],
                    "arrayminus": [estimate - lower],
                    "color": DARK_TEXT,
                    "thickness": 1.6,
                    "width": 5,
                },
                showlegend=False,
                hovertemplate=(
                    f"{label}<br>κ={estimate:.3f}"
                    f"<br>95% CI [{lower:.3f}, {upper:.3f}]<extra></extra>"
                ),
            ),
            row=1,
            col=2,
        )

    ecdf_specs = [
        ("Overall", kendall_data, "rgb(23,107,135)", "solid"),
        (
            "Well-studied",
            kendall_data[kendall_data["StudyStatus"] == "well_studied"],
            "rgb(102,157,183)",
            "dash",
        ),
        (
            "Uncharacterized",
            kendall_data[kendall_data["StudyStatus"] == "uncharacterized"],
            ACCENT_COLOR,
            "dot",
        ),
    ]
    for label, rows, color, dash in ecdf_specs:
        values = np.sort(rows["KendallW"].to_numpy(dtype=float))
        cumulative = np.arange(1, len(values) + 1) / len(values)
        figure.add_trace(
            go.Scatter(
                x=values,
                y=cumulative,
                mode="lines",
                line={"color": color, "width": 3, "dash": dash, "shape": "hv"},
                name=f"Kendall W: {label}",
                hovertemplate=(
                    f"{label}<br>Kendall's W: %{{x:.3f}}"
                    "<br>Cumulative genes: %{y:.1%}<extra></extra>"
                ),
            ),
            row=2,
            col=2,
        )

    top1_index = top1_data.set_index(["ScopeID", "Top1AgreementPattern"])
    top1_labels = [f"{label} (n={n_genes})" for _, label, n_genes in TOP1_SCOPE_ORDER]
    for pattern, pattern_label, color in TOP1_PATTERNS:
        fractions = np.array(
            [top1_index.loc[(scope_id, pattern), "Fraction"] for scope_id, _, _ in TOP1_SCOPE_ORDER],
            dtype=float,
        )
        lower = np.array(
            [top1_index.loc[(scope_id, pattern), "FractionCILower"] for scope_id, _, _ in TOP1_SCOPE_ORDER],
            dtype=float,
        )
        upper = np.array(
            [top1_index.loc[(scope_id, pattern), "FractionCIUpper"] for scope_id, _, _ in TOP1_SCOPE_ORDER],
            dtype=float,
        )
        figure.add_trace(
            go.Bar(
                x=100 * fractions,
                y=top1_labels,
                orientation="h",
                name=f"Top-1: {pattern_label}",
                marker={"color": color, "line": {"color": "white", "width": 0.7}},
                text=[f"{100 * value:.1f}%" if value >= 0.05 else "" for value in fractions],
                textposition="inside",
                customdata=np.column_stack([100 * lower, 100 * upper]),
                hovertemplate=(
                    f"{pattern_label}: %{{x:.1f}}%"
                    "<br>95% CI: [%{customdata[0]:.1f}, %{customdata[1]:.1f}]%"
                    "<extra></extra>"
                ),
            ),
            row=3,
            col=2,
        )

    top1_sums = top1_data[
        top1_data["ScopeID"].isin([scope for scope, _, _ in TOP1_SCOPE_ORDER])
    ].groupby("ScopeID")["Fraction"].sum()
    if not np.allclose(top1_sums, 1.0, atol=1e-12):
        raise ValueError("Top-1 consensus fractions must sum to one.")

    overall_ordinal = ordinal_data.set_index("ScopeID").loc["overall"]
    figure.add_vline(
        x=0,
        line_color="rgb(70,74,77)",
        line_dash="dash",
        line_width=1.5,
        row=1,
        col=2,
    )
    figure.update_xaxes(
        title_text="Experts (%)",
        range=[0, 128],
        gridcolor=GRID_COLOR,
        row=1,
        col=1,
    )
    figure.update_yaxes(
        type="multicategory",
        autorange="reversed",
        automargin=True,
        tickfont={"size": 11},
        row=1,
        col=1,
    )
    figure.update_xaxes(
        title_text="Fleiss-κ (95% CI)",
        range=[-0.08, 0.24],
        gridcolor=GRID_COLOR,
        zeroline=False,
        row=1,
        col=2,
    )
    figure.update_yaxes(
        categoryorder="array",
        categoryarray=kappa_labels,
        autorange="reversed",
        automargin=True,
        tickfont={"size": 12},
        row=1,
        col=2,
    )
    figure.update_xaxes(
        title_text="Kendall's W",
        range=[0, 1],
        gridcolor=GRID_COLOR,
        row=2,
        col=2,
    )
    figure.update_yaxes(
        title_text="Cumulative genes (%)",
        range=[0, 1],
        tickformat=".0%",
        gridcolor=GRID_COLOR,
        row=2,
        col=2,
    )
    figure.update_xaxes(
        title_text="Genes (%)",
        range=[0, 100],
        gridcolor=GRID_COLOR,
        row=3,
        col=2,
    )
    figure.update_yaxes(
        autorange="reversed",
        automargin=True,
        tickfont={"size": 12},
        row=3,
        col=2,
    )

    figure.add_annotation(
        x=0.25,
        y=1.035,
        xref="paper",
        yref="paper",
        text=(
            "120 experts · 200 genes · 600 expert–gene rankings · "
            "5 genes/expert · 3 experts/gene"
        ),
        showarrow=False,
        font={"size": 15, "color": DARK_TEXT},
    )
    figure.add_annotation(
        x=0,
        y=-0.055,
        xref="paper",
        yref="paper",
        xanchor="left",
        align="left",
        text=(
            "† Multi-select dimensions use all 120 experts as the denominator; "
            "percentages need not sum to 100%. Other percentages use nonmissing "
            "experts (annotation experience n=109; AI experience n=101)."
        ),
        showarrow=False,
        font={"size": 12, "color": DARK_TEXT},
    )
    figure.add_annotation(
        x=0,
        y=-0.085,
        xref="paper",
        yref="paper",
        xanchor="left",
        text="No conflicts of interest were declared by any of the 120 experts.",
        showarrow=False,
        font={"size": 12, "color": DARK_TEXT},
    )
    figure.add_annotation(
        x=0,
        y=1.10,
        xref="x2 domain",
        yref="y2 domain",
        xanchor="left",
        align="left",
        text=(
            "Items: gene × model; rank categories R1–R5; 3 ratings/item.<br>"
            "95% percentile CI from 10,000 stratified gene-block bootstraps; "
            "45 exploratory scopes remain in the full 58-row table."
        ),
        showarrow=False,
        font={"size": 11, "color": DARK_TEXT},
    )
    figure.add_annotation(
        x=0,
        y=1.14,
        xref="x3 domain",
        yref="y3 domain",
        xanchor="left",
        align="left",
        text=(
            f"Overall mean W={overall_ordinal['KendallWMean']:.3f} "
            f"(95% CI {overall_ordinal['KendallWMeanCILower']:.3f}–"
            f"{overall_ordinal['KendallWMeanCIUpper']:.3f}); "
            "CI describes the mean, not the ECDF."
        ),
        showarrow=False,
        font={"size": 11, "color": DARK_TEXT},
    )
    for label, x, y in [("a", -0.035, 1.015), ("b", 0.57, 1.015), ("c", 0.57, 0.645), ("d", 0.57, 0.315)]:
        figure.add_annotation(
            x=x,
            y=y,
            xref="paper",
            yref="paper",
            text=f"<b>{label}</b>",
            showarrow=False,
            font={"size": 24, "color": DARK_TEXT},
        )

    figure.update_layout(
        barmode="stack",
        plot_bgcolor="white",
        paper_bgcolor="white",
        font_family="Arial",
        font_color=DARK_TEXT,
        font_size=14,
        width=2800,
        height=2400,
        margin={"l": 240, "r": 100, "t": 150, "b": 230},
        legend={
            "orientation": "h",
            "x": 0.58,
            "xanchor": "left",
            "y": -0.105,
            "yanchor": "top",
            "font": {"size": 11},
        },
    )
    for annotation in figure.layout.annotations[:4]:
        annotation.font = {"family": "Arial", "size": 18, "color": DARK_TEXT}
    return figure

## Fig. 2d-f

The overall panels preserve the original rank composition and add primary
crossed bootstrap intervals where they map naturally to the plotted statistic.

In [ ]:
fig_2d = rank_distribution_figure("overall")
render_figure(fig_2d, "fig.2d.phytobench-gene.percent.bar")

fig_2e = pairwise_probability_figure("overall")
render_figure(fig_2e, "fig.2e.phytobench-gene.prob.heatmap")

fig_2f = elo_score_figure("overall", y_range=(1200, 1700))
render_figure(fig_2f, "fig.2f.phytobench-gene.score.bar")

## Supplementary Fig. 10-13

Supplementary Fig. 10 reports stratified rank compositions, Supplementary
Fig. 11 reports panel composition and agreement, Supplementary Fig. 12 reports
stratified pairwise probabilities with confidence intervals, and Supplementary
Fig. 13 reports stratified Elo-like scores with confidence intervals.

In [ ]:
for scope in STRATIFIED_SCOPES:
    render_figure(
        rank_distribution_figure(scope),
        f"supplementary_fig.10.phytobench-gene.{scope}.percent.bar",
    )

render_figure(
    expert_agreement_figure(),
    "supplementary_fig.11.expert-panel-and-agreement",
)

for scope in STRATIFIED_SCOPES:
    render_figure(
        pairwise_probability_figure(scope),
        f"supplementary_fig.12.phytobench-gene.{scope}.prob.heatmap",
    )
    render_figure(
        elo_score_figure(scope),
        f"supplementary_fig.13.phytobench-gene.{scope}.score.bar",
    )